# Pilote BEV : LiDAR seul et fusion camera/LiDAR

Deux runs distincts sur les memes 410 frames train et 102 frames validation. Encodeurs geles, nouveaux blocs initialises a l'identique, inputs complets et aucun depth holdout. Le controle LiDAR conserve l'architecture commune, mais coupe les features/supports camera. Ne jamais reprendre les poids du diagnostic quatre frames. Run All ne lance pas d'entrainement par defaut. La validation par frame n'est pas un test de scenes independantes ni une AP 3D.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / 'src/rgb_lidar_fusion').is_dir() and (p / 'pyproject.toml').is_file()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from the repository or notebooks directory.')
sys.path.insert(0, str(ROOT / 'src'))
import torch
from rgb_lidar_fusion.bev_pilot import BEVFusionPilotConfig, VARIANTS, prepare_bev_fusion_pilot_pair, read_bev_pilot_split
from rgb_lidar_fusion.bev_trainval import validate_bev_pilot_resume
from rgb_lidar_fusion.bev_pilot_analysis import (
    load_bev_pilot_run, plot_bev_pilot_curves, compare_bev_pilot_runs, analyze_bev_pilot_validation)
from rgb_lidar_fusion.bev_analysis import plot_bev_fusion_sample
from rgb_lidar_fusion.lidar_center_training import _write_json
print(f'ROOT={ROOT}')

## Parametres du run selectionne
Commencer par lidar_only, puis fusion, sans les lancer simultanement. Chaque variante vise 10 epochs, batch 1 et FP32. EPOCHS est le total vise lors d'une reprise. DEVICE ne concerne que le processus enfant ; les audits, courbes et inspections restent CPU. La configuration effectivement resolue est enregistree puis fournie au CLI, pour ne pas perdre les surcharges du notebook.

In [ ]:
RUN_TRAINING = False
RESUME_EXISTING = False
RUN_COMPARISON = False
VARIANT = 'lidar_only'
DEVICE = 'cuda'
EPOCHS = 10
VALIDATION_SAMPLE_POSITION = 0
INSPECT_SELECTION = 'best'
CONFIG_PATH = ROOT / 'configs/training/kitti_bev_fusion_pilot.json'
OUTPUT_ROOT = ROOT / 'results/training/kitti_bev_fusion_pilot_512'
ANALYSIS_DIR = ROOT / 'results/analysis/bev_fusion' / OUTPUT_ROOT.name
values = json.loads(CONFIG_PATH.read_text())
for key in ('data_root', 'split_path', 'camera_checkpoint', 'lidar_checkpoint'):
    values[key] = str(ROOT / values[key])
values.update(output_dir=str(OUTPUT_ROOT), device=DEVICE, epochs=EPOCHS)
config = BEVFusionPilotConfig.from_mapping(values)
RUN_DIR = config.output_for_variant(VARIANT)
CHECKPOINT_PATH = RUN_DIR / 'checkpoints/latest.pt'
LOG_PATH = OUTPUT_ROOT.parent / f'{OUTPUT_ROOT.name}_{VARIANT}.jsonl'
LAUNCH_CONFIG_PATH = ANALYSIS_DIR / f'launch_{VARIANT}.json'
torch.set_num_threads(config.cpu_threads)
print(config.serialized())
print(f'variant={VARIANT} training={RUN_TRAINING} resume={RESUME_EXISTING} total_epochs={EPOCHS}')
print(f'log={LOG_PATH}')

## Audit CPU du split et des poids
Un demarrage neuf controle les deux variantes : memes IDs, empreintes et sources, meme hash des nouveaux blocs. La reprise controle le checkpoint pilote selectionne et les fichiers actuels, sans rouvrir les anciens checkpoints d'encodeurs. Le runner repetera ses controles et le preflight dans le processus enfant.

In [ ]:
split = read_bev_pilot_split(config)
print(f"train={len(split['train_ids'])} validation={len(split['validation_ids'])} overlap={len(set(split['train_ids']) & set(split['validation_ids']))}")
if RESUME_EXISTING:
    resume_model, resume_config, resume_payload = load_bev_pilot_run(CHECKPOINT_PATH)
    validate_bev_pilot_resume(resume_payload, config=config, variant=VARIANT, split=split)
    print(f"Resume epoch={resume_payload['epoch']} step={resume_payload['step']}")
    print(resume_payload['training_metadata']['initialization'])
    del resume_model, resume_payload
else:
    starts = prepare_bev_fusion_pilot_pair(config)
    for name, start in starts.items():
        print(name, start.initialization, start.pretraining['camera']['provenance'])
    del starts

## Entrainement explicite d'une seule variante
Apres l'audit, passer RUN_TRAINING a True, reexecuter les parametres et cette cellule. CUDA exige 2048 MiB libres ; aucun repli CPU ou retry. Le dossier de la variante doit etre neuf, sauf reprise explicite de son latest.pt. Evaluations et checkpoints a l'epoch 0 puis apres chaque epoch complete. Interrupt termine le processus enfant, puis kill si necessaire ; sa sortie libere ses ressources GPU. Les mises a jour depuis la derniere epoch sauvegardee seront perdues. La cellule affiche les phases, processed/total et les metriques ; le JSONL est aussi sauvegarde hors du dossier de la variante.

In [ ]:
if RUN_TRAINING:
    if not RESUME_EXISTING and RUN_DIR.exists() and any(RUN_DIR.iterdir()):
        raise ValueError('Variant output directory is not empty; choose a new pilot root or resume.')
    if RESUME_EXISTING and not CHECKPOINT_PATH.is_file():
        raise FileNotFoundError(CHECKPOINT_PATH)
    LAUNCH_CONFIG_PATH.parent.mkdir(parents=True, exist_ok=True)
    _write_json(LAUNCH_CONFIG_PATH, config.serialized())
    command = [sys.executable, str(ROOT / 'scripts/train_bev_pilot.py'),
               '--config', str(LAUNCH_CONFIG_PATH), '--variant', VARIANT]
    if RESUME_EXISTING:
        command += ['--resume-from', str(CHECKPOINT_PATH)]
    env = dict(os.environ, CUDA_LAUNCH_BLOCKING='1', PYTHONUNBUFFERED='1')
    env['PYTHONPATH'] = os.pathsep.join(filter(None, (str(ROOT / 'src'), env.get('PYTHONPATH'))))
    LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
    with LOG_PATH.open('a' if RESUME_EXISTING else 'w') as log, subprocess.Popen(
        command, cwd=ROOT, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1) as process:
        try:
            for line in process.stdout:
                log.write(line)
                log.flush()
                print(line.rstrip(), flush=True)
            if process.wait() != 0:
                raise RuntimeError(f'Training process failed with code {process.returncode}.')
        except BaseException:
            if process.poll() is None:
                process.terminate()
                try:
                    process.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait()
            raise
else:
    print('Training disabled. Set RUN_TRAINING=True only after reviewing the variant and audit.')

## Courbes des epochs terminees
Lire les latest.pt, sources autoritatives, plutot qu'un JSON potentiellement interrompu. Un seul run ou des runs partiels peuvent etre affiches. Deux courbes ne sont superposees que si les protocoles sont compatibles ; des avancements differents ne constituent pas un bilan comparatif final. L'erreur XY ne concerne que les centres apparies : None signifie aucun appariement, pas une erreur nulle. Le PNG des courbes est sauvegarde.

In [ ]:
runs = {}
for name in VARIANTS:
    path = config.output_for_variant(name) / 'checkpoints/latest.pt'
    if path.is_file():
        cpu_model, saved_config, payload = load_bev_pilot_run(path)
        runs[name] = payload
        print(f"{name}: completed_epoch={payload['epoch']} step={payload['step']} best_epoch={payload['best']['epoch']}")
        del cpu_model
if runs:
    fig = plot_bev_pilot_curves(runs)
    ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
    figure_path = ANALYSIS_DIR / 'training_curves.png'
    fig.savefig(figure_path, dpi=150, bbox_inches='tight')
    print(f'Saved figure: {figure_path}')
    display(fig)
    plt.close(fig)
else:
    print('No pilot checkpoint yet.')

## Comparaison finale controlee
Passer RUN_COMPARISON a True uniquement apres les deux runs. Le controle exige le meme budget termine, le split/contenu identique, les memes poids sources et initialisations, le meme apprentissage et les memes regles d'evaluation. Il distingue les metriques finales a epoch egale des meilleurs checkpoints selectionnes independamment par val_loss. Le best peut etre a l'epoch 0. Ce sont des scores de validation ayant servi a la selection, pas un test independant. Un resultat neutre ou negatif pour la fusion reste un resultat valide. Le JSON est sauvegarde.

In [ ]:
if RUN_COMPARISON:
    comparison = compare_bev_pilot_runs(
        config.output_for_variant('lidar_only') / 'checkpoints/latest.pt',
        config.output_for_variant('fusion') / 'checkpoints/latest.pt')
    ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
    comparison_path = ANALYSIS_DIR / 'comparison.json'
    _write_json(comparison_path, comparison)
    display(comparison)
    print(f'Saved comparison: {comparison_path}')
else:
    print('Final comparison disabled. Complete both runs before setting RUN_COMPARISON=True.')

## Inspection CPU d'une frame de validation
VARIANT choisit le run ; VALIDATION_SAMPLE_POSITION indexe uniquement ses IDs de validation sauvegardes. INSPECT_SELECTION vaut best ou latest. Les titres et noms PNG indiquent la variante, la selection et l'epoch des poids effectivement utilises. Changer de position pour chercher reussites, faux positifs et objets manques ; le notebook ne remplace jamais la validation par une frame apprise. RMS reste une amplitude de features, pas une confiance semantique. Dans le controle LiDAR, les panneaux camera sont volontairement vides.

In [ ]:
if CHECKPOINT_PATH.is_file():
    inspection = analyze_bev_pilot_validation(
        CHECKPOINT_PATH, validation_position=VALIDATION_SAMPLE_POSITION, selection=INSPECT_SELECTION)
    sample = inspection.sample
    print(f'Validation ID={sample.sample_id} variant={inspection.variant} selection={inspection.selection} epoch={inspection.checkpoint_epoch}')
    print(sample.loss, sample.matching)
    fig = plot_bev_fusion_sample(sample, grid=inspection.config.model_config.grid,
        weights_label=f'{inspection.variant}, {inspection.selection}, epoch {inspection.checkpoint_epoch}')
    ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
    figure_path = ANALYSIS_DIR / f'{inspection.variant}_{inspection.selection}_epoch_{inspection.checkpoint_epoch:04d}_{sample.sample_id}.png'
    fig.savefig(figure_path, dpi=150, bbox_inches='tight')
    print(f'Saved figure: {figure_path}')
    display(fig)
    plt.close(fig)
else:
    print('No checkpoint for the selected variant yet.')